# MB-CBA: Multi-Band CNN–BiLSTM–Attention untuk Deteksi Kognitif EEG
### Pipeline lengkap: Kaggle dataset → Multi-band preprocessing → LOSO evaluation

**Perubahan utama dari notebook asli:**
- Data `.mat` dibaca langsung dari Kaggle (14 channel, sama seperti notebook asli)
- Ditambahkan **multi-band decomposition** (delta/theta/alpha/SMR/beta) → input (14×5=70, 256)
- Evaluasi diubah dari random split → **LOSO cross-validation** (cross-subject)
- Model ditambahkan **attention mechanism**
- Semua metrik Q1 dilaporkan: accuracy, weighted F1, macro F1, per-class F1, confusion matrix


In [ ]:
!pip install kagglehub scipy scikit-learn seaborn tqdm -q

import kagglehub, os, json, warnings
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from scipy.signal import butter, filtfilt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, precision_score, recall_score
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


## 1. Load Dataset dari Kaggle

In [ ]:
# Download dataset Aci et al. 2019
path = kagglehub.dataset_download('inancigdem/eeg-data-for-mental-attention-state-detection')
data_root = os.path.join(path, 'EEG Data')
all_files = sorted([f for f in os.listdir(data_root) if f.endswith('.mat')])
print(f'Total file .mat: {len(all_files)}')
print('Contoh file:', all_files[:5])


Total file .mat: 34
Contoh file: ['eeg_record1.mat', 'eeg_record10.mat', 'eeg_record11.mat', 'eeg_record12.mat', 'eeg_record13.mat']


## 2. Konfigurasi

In [ ]:
CFG = {
    # Dataset
    'fs'          : 128,       # sampling rate (Hz)
    'n_subjects'  : 5,         # 5 subjek
    'n_sessions'  : 7,         # 7 sesi total per subjek
    'n_habituasi' : 2,         # 2 sesi pertama = habituasi (dibuang)

    # 14 channel sesuai dataset asli (Emotiv Epoc, modifikasi)
    # Catatan: paper Aci et al menggunakan 7 channel aktif (F3,Fz,F4,C3,Cz,C4,Pz)
    # tapi file .mat memiliki 14 channel Emotiv standar
    'channels': ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4'],
    'n_channels'  : 14,

    # Fase label (menit) — sesuai protokol Aci et al. 2019
    'phases': [
        (0,  10, 0),   # Focused   0–10 mnt
        (10, 20, 1),   # Unfocused 10–20 mnt
        (20, 30, 2),   # Drowsy    20–30 mnt
    ],
    'class_names' : ['Focused', 'Unfocused', 'Drowsy'],

    # Band frekuensi NON-OVERLAPPING (gold standard)
    'bands': {
        'delta': (0.5, 4),
        'theta': (4,   8),
        'alpha': (8,  12),
        'smr'  : (12, 15),
        'beta' : (15, 30),
    },

    # Epoch
    'epoch_len_s' : 2,         # panjang epoch (detik)
    'step_s'      : 0.25,      # sliding window step (detik)
    'amp_thresh'  : 150,       # threshold artefak µV (sebelum normalisasi)

    # Training
    'lr'          : 1e-3,
    'batch_size'  : 32,
    'max_epochs'  : 30,
    'patience'    : 7,
    'dropout'     : 0.3,
    'bilstm_h'    : 64,
    'save_dir'    : 'saved_models',
    'n_classes'   : 3,
}

CFG['T']     = CFG['fs'] * CFG['epoch_len_s']       # 256
CFG['step']  = int(CFG['fs'] * CFG['step_s'])        # 32
CFG['n_bands'] = len(CFG['bands'])                   # 5
CFG['C_mb']  = CFG['n_channels'] * CFG['n_bands']   # 70

os.makedirs(CFG['save_dir'], exist_ok=True)
print(f"Input tensor per epoch: ({CFG['C_mb']}, {CFG['T']}) = ({CFG['n_channels']} ch x {CFG['n_bands']} bands, {CFG['T']} samples)")


Input tensor per epoch: (70, 256) = (14 ch x 5 bands, 256 samples)


## 3. Pembacaan File .mat (format Emotiv Epoc)

In [ ]:
COLUMNS_MAT = [
    'ED_COUNTER','ED_INTERPOLATED','ED_RAW_CQ',
    'ED_AF3','ED_F7','ED_F3','ED_FC5','ED_T7','ED_P7',
    'ED_O1','ED_O2','ED_P8','ED_T8','ED_FC6','ED_F4',
    'ED_F8','ED_AF4','ED_GYROX','ED_GYROY','ED_TIMESTAMP',
    'ED_ES_TIMESTAMP','ED_FUNC_ID','ED_FUNC_VALUE',
    'ED_MARKER','ED_SYNC_SIGNAL'
]
EEG_COLS_RAW = ['ED_AF3','ED_F7','ED_F3','ED_FC5','ED_T7','ED_P7',
                 'ED_O1','ED_O2','ED_P8','ED_T8','ED_FC6','ED_F4',
                 'ED_F8','ED_AF4']

def load_mat_session(mat_path):
    """
    Baca satu file .mat dari dataset Aci et al.
    Return: np.ndarray shape (n_channels=14, n_samples) — unit asli µV
    """
    mat  = scipy.io.loadmat(mat_path)
    data = mat['o']['data'][0, 0]
    df   = pd.DataFrame(data, columns=COLUMNS_MAT)

    # Buang baris interpolated
    df = df[df['ED_INTERPOLATED'] == 0]

    eeg = df[EEG_COLS_RAW].values.T.astype(np.float64)  # (14, n_samples)
    return eeg

# Test baca satu file
_test = load_mat_session(os.path.join(data_root, all_files[0]))
print(f'Shape satu sesi: {_test.shape}  ({_test.shape[1]/CFG["fs"]/60:.1f} menit)')
print(f'Amplitude range: [{_test.min():.1f}, {_test.max():.1f}] µV')


Shape satu sesi: (14, 308868)  (40.2 menit)
Amplitude range: [547.2, 7803.1] µV


## 4. Multi-Band Decomposition (Gold Standard, Non-Overlapping)

In [ ]:
def bandpass_filter(signal, lo, hi, fs, order=4):
    """Zero-phase Butterworth bandpass. signal: (C, N) atau (N,)"""
    nyq  = fs / 2.0
    high = min(hi / nyq, 0.99)
    b, a = butter(order, [lo / nyq, high], btype='band')
    if signal.ndim == 1:
        return filtfilt(b, a, signal)
    return np.array([filtfilt(b, a, signal[c]) for c in range(signal.shape[0])])

def multiband_decompose(eeg, cfg):
    """
    eeg  : (n_ch, n_samples)
    return: (n_ch * n_bands, n_samples) = (70, n_samples)
    Band order: delta, theta, alpha, SMR, beta (non-overlapping)
    """
    parts = []
    for band_name, (lo, hi) in cfg['bands'].items():
        parts.append(bandpass_filter(eeg, lo, hi, cfg['fs']))
    return np.concatenate(parts, axis=0)   # (70, n_samples)

# Verifikasi
_mb = multiband_decompose(_test, CFG)
print(f'Multi-band shape: {_mb.shape}')
print('Band breakdown (14 ch x 5 bands = 70):')
for i, (b, (lo, hi)) in enumerate(CFG['bands'].items()):
    print(f'  [{i*14}:{(i+1)*14}] {b.upper():6s} {lo:4.1f}–{hi:2d} Hz')


Multi-band shape: (70, 308868)
Band breakdown (14 ch x 5 bands = 70):
  [0:14] DELTA   0.5– 4 Hz
  [14:28] THETA   4.0– 8 Hz
  [28:42] ALPHA   8.0–12 Hz
  [42:56] SMR    12.0–15 Hz
  [56:70] BETA   15.0–30 Hz


In [ ]:
def normalize_epoch(epoch):
    """Per-epoch, per-channel z-score. epoch: (C_mb, T)"""
    mu  = epoch.mean(axis=1, keepdims=True)
    std = epoch.std(axis=1,  keepdims=True)
    return (epoch - mu) / (std + 1e-8)

def segment_and_label(eeg_mb, cfg):
    """
    eeg_mb : (C_mb, n_samples) — satu sesi, sudah multi-band
    return : epochs (N, C_mb, T), labels (N,)
    Label: 0=Focused 1=Unfocused 2=Drowsy — berdasarkan posisi temporal
    """
    T, step, fs = cfg['T'], cfg['step'], cfg['fs']
    epochs, labels = [], []
    n_total = eeg_mb.shape[1]

    start = 0
    while start + T <= n_total:
        end   = start + T
        epoch = eeg_mb[:, start:end]

        # Posisi tengah epoch dalam menit (sebelum normalisasi)
        mid_min = ((start + end) / 2) / fs / 60.0

        label = None
        for (t0, t1, lbl) in cfg['phases']:
            if t0 <= mid_min < t1:
                label = lbl
                break

        if label is not None:
            epoch_norm = normalize_epoch(epoch)
            if not (np.isnan(epoch_norm).any() or np.isinf(epoch_norm).any()):
                epochs.append(epoch_norm.astype(np.float32))
                labels.append(label)

        start += step

    if not epochs:
        return np.empty((0, cfg['C_mb'], T), dtype=np.float32), np.empty(0, dtype=np.int64)
    return np.stack(epochs), np.array(labels, dtype=np.int64)

print('Fungsi segmentasi dan labeling siap.')


Fungsi segmentasi dan labeling siap.


## 5. Grouping File per Subjek & Build X_subjects, y_subjects

In [ ]:
# Dataset Aci et al: 5 subjek x 7 sesi = 35 file
# File diurutkan alphabetically → S1 sesi1..7, S2 sesi1..7, dst.
# 2 sesi pertama per subjek = habituasi (dibuang)

def build_subject_groups(all_files, cfg):
    """
    Kelompokkan file per subjek.
    Asumsi file sudah terurut: S1S1, S1S2, ..., S1S7, S2S1, ...
    Return: dict {sub_idx: [path_sesi3, path_sesi4, ..., path_sesi7]}
    """
    n_per_sub = cfg['n_sessions']        # 7
    skip      = cfg['n_habituasi']       # 2
    groups    = {}
    for s in range(cfg['n_subjects']):
        start = s * n_per_sub + skip     # skip 2 habituasi
        end   = s * n_per_sub + n_per_sub
        groups[s] = all_files[start:end]
    return groups

groups = build_subject_groups(all_files, CFG)
for s, flist in groups.items():
    print(f'Subject {s+1}: {len(flist)} sesi → {flist}')


Subject 1: 5 sesi → ['eeg_record11.mat', 'eeg_record12.mat', 'eeg_record13.mat', 'eeg_record14.mat', 'eeg_record15.mat']
Subject 2: 5 sesi → ['eeg_record18.mat', 'eeg_record19.mat', 'eeg_record2.mat', 'eeg_record20.mat', 'eeg_record21.mat']
Subject 3: 5 sesi → ['eeg_record24.mat', 'eeg_record25.mat', 'eeg_record26.mat', 'eeg_record27.mat', 'eeg_record28.mat']
Subject 4: 5 sesi → ['eeg_record30.mat', 'eeg_record31.mat', 'eeg_record32.mat', 'eeg_record33.mat', 'eeg_record34.mat']
Subject 5: 4 sesi → ['eeg_record6.mat', 'eeg_record7.mat', 'eeg_record8.mat', 'eeg_record9.mat']


In [ ]:
def build_dataset(groups, data_root, cfg):
    """
    Return:
        X_subjects: list[5] of (N_i, C_mb=70, T=256) float32
        y_subjects: list[5] of (N_i,) int64
    """
    X_subjects, y_subjects = [], []

    for sub_idx, file_list in groups.items():
        print(f'\nSubject {sub_idx+1}/{cfg["n_subjects"]}')
        sub_X, sub_y = [], []

        for fname in file_list:
            fpath = os.path.join(data_root, fname)
            # 1. Load raw EEG
            eeg_raw = load_mat_session(fpath)             # (14, N)
            # 2. Multi-band decomposition
            eeg_mb  = multiband_decompose(eeg_raw, cfg)   # (70, N)
            # 3. Segmentasi + labelling
            X, y    = segment_and_label(eeg_mb, cfg)
            if len(X) > 0:
                sub_X.append(X)
                sub_y.append(y)
                counts = np.bincount(y, minlength=3)
                print(f'  {fname}: {X.shape[0]} epochs | '
                      f'F={counts[0]} U={counts[1]} D={counts[2]}')

        X_sub = np.concatenate(sub_X, axis=0)
        y_sub = np.concatenate(sub_y, axis=0)
        counts = np.bincount(y_sub, minlength=3)
        print(f'  → Total: {X_sub.shape} | F={counts[0]} U={counts[1]} D={counts[2]}')
        X_subjects.append(X_sub)
        y_subjects.append(y_sub)

    return X_subjects, y_subjects

X_subjects, y_subjects = build_dataset(groups, data_root, CFG)
print(f'\n✓ Dataset siap: {CFG["n_subjects"]} subjek')
print(f'  Shape contoh: X={X_subjects[0].shape}, y={y_subjects[0].shape}')



Subject 1/5
  eeg_record11.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record12.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record13.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record14.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record15.mat: 7196 epochs | F=2396 U=2400 D=2400
  → Total: (35980, 70, 256) | F=11980 U=12000 D=12000

Subject 2/5
  eeg_record18.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record19.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record2.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record20.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record21.mat: 7196 epochs | F=2396 U=2400 D=2400
  → Total: (35980, 70, 256) | F=11980 U=12000 D=12000

Subject 3/5
  eeg_record24.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record25.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record26.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record27.mat: 7196 epochs | F=2396 U=2400 D=2400
  eeg_record28.mat: 6697 epochs | F=2396 U=2400 D=1901
  → Total: (35481, 70, 256) |

## 6. Diagnosis Data Pipeline

In [ ]:
def diagnose(X_subjects, y_subjects, cfg):
    print('='*55)
    all_ok = True
    for i, (X, y) in enumerate(zip(X_subjects, y_subjects)):
        print(f'Subject {i+1}:')
        # Shape
        ok = X.ndim==3 and X.shape[1]==cfg['C_mb'] and X.shape[2]==cfg['T']
        print(f'  {"✓" if ok else "✗"} Shape: {X.shape} (expect N,{cfg["C_mb"]},{cfg["T"]})')
        if not ok: all_ok = False
        # Labels
        counts = np.bincount(y, minlength=3)
        ok2 = (counts>0).all()
        print(f'  {"✓" if ok2 else "✗"} Labels: F={counts[0]} U={counts[1]} D={counts[2]}')
        if not ok2: all_ok = False
        # Norm
        ok3 = 0.3 < X.std() < 5.0
        print(f'  {"✓" if ok3 else "✗"} Std: {X.std():.3f} (expect 0.3–5.0)')
        if not ok3: all_ok = False
        # NaN
        ok4 = not (np.isnan(X).any() or np.isinf(X).any())
        print(f'  {"✓" if ok4 else "✗"} NaN/Inf: {"bersih" if ok4 else "ADA MASALAH"}')
        if not ok4: all_ok = False
    print('='*55)
    print(f'{"✓ Semua OK — siap training" if all_ok else "✗ Ada masalah — periksa preprocessing"}')
    return all_ok

diagnose(X_subjects, y_subjects, CFG)


Subject 1:
  ✓ Shape: (35980, 70, 256) (expect N,70,256)
  ✓ Labels: F=11980 U=12000 D=12000
  ✓ Std: 0.989 (expect 0.3–5.0)
  ✓ NaN/Inf: bersih
Subject 2:
  ✓ Shape: (35980, 70, 256) (expect N,70,256)
  ✓ Labels: F=11980 U=12000 D=12000
  ✓ Std: 0.991 (expect 0.3–5.0)
  ✓ NaN/Inf: bersih
Subject 3:
  ✓ Shape: (35481, 70, 256) (expect N,70,256)
  ✓ Labels: F=11980 U=12000 D=11501
  ✓ Std: 0.995 (expect 0.3–5.0)
  ✓ NaN/Inf: bersih
Subject 4:
  ✓ Shape: (35980, 70, 256) (expect N,70,256)
  ✓ Labels: F=11980 U=12000 D=12000
  ✓ Std: 0.998 (expect 0.3–5.0)
  ✓ NaN/Inf: bersih
Subject 5:
  ✓ Shape: (28784, 70, 256) (expect N,70,256)
  ✓ Labels: F=9584 U=9600 D=9600
  ✓ Std: 0.990 (expect 0.3–5.0)
  ✓ NaN/Inf: bersih
✓ Semua OK — siap training


True

## 7. Definisi Model MB-CBA

In [ ]:
class Attention(nn.Module):
    """Additive (Bahdanau) attention."""
    def __init__(self, hidden_dim, attn_dim=64):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, H):
        # H: (batch, T', 2h)
        e     = self.v(torch.tanh(self.W(H))).squeeze(-1)  # (batch, T')
        alpha = torch.softmax(e, dim=-1)                    # (batch, T')
        ctx   = (alpha.unsqueeze(-1) * H).sum(dim=1)       # (batch, 2h)
        return ctx, alpha


class MBCBA(nn.Module):
    """Multi-Band CNN-BiLSTM-Attention."""
    def __init__(self, cfg):
        super().__init__()
        C_mb = cfg['C_mb']   # 70
        h    = cfg['bilstm_h']
        p    = cfg['dropout']
        K    = cfg['n_classes']

        # CNN Block 1: (B, C_mb, T) → (B, 64, T/2)
        self.conv1 = nn.Sequential(
            nn.Conv1d(C_mb, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
        )
        # CNN Block 2: (B, 64, T/2) → (B, 128, T/2)
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p),
        )
        # BiLSTM
        self.bilstm = nn.LSTM(128, h, batch_first=True, bidirectional=True)
        # Attention
        self.attention = Attention(2*h, attn_dim=64)
        # Classifier
        self.classifier = nn.Sequential(nn.Dropout(p), nn.Linear(2*h, K))

    def forward(self, x):
        # x: (B, C_mb, T)
        h = self.conv1(x)             # (B, 64, T/2)
        h = self.conv2(h)             # (B, 128, T/2)
        h = h.permute(0, 2, 1)       # (B, T/2, 128)
        h, _ = self.bilstm(h)         # (B, T/2, 2*bilstm_h)
        ctx, alpha = self.attention(h)
        out = self.classifier(ctx)    # (B, K)
        return out, alpha

# Hitung total parameter
_m = MBCBA(CFG)
total_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')
print(f'Input shape: (batch, {CFG["C_mb"]}, {CFG["T"]})')


Total trainable parameters: 146,627
Input shape: (batch, 70, 256)


## 8. LOSO Cross-Validation dengan MB-CBA
**Mengapa Proses LOSO dengan MB-CBA Sangat Lama?**
Ada beberapa faktor yang menyebabkan proses berjalan sangat lama (sudah 1.5 jam belum selesai):

1. Ukuran Dataset Sangat Besar
**Dari output sebelumnya:**
- Subject 1: 35,980 epochs × 70 channels × 256 timesteps
- Subject 2: 35,980 epochs
- Subject 3: 35,481 epochs  
- Subject 4: 35,980 epochs
- Subject 5: 28,784 epochs

**Total**: ~172,000 epochs untuk 5 subjek
**Setiap epoch** = 70 × 256 = 17,920 nilai floating point

2. LOSO Melatih Model 5 Kali
Untuk setiap fold (subjek test):
- Training: 4 subjek lain ≈ 140,000 epochs
- Validation: 20% dari training ≈ 28,000 epochs
- Per epoch training: 140,000 / 32 batch_size ≈ 4,375 batch per epoch
- 30 epochs × 4,375 batch = 131,250 batch per fold
- 5 folds × 131,250 = 656,250 total batch

3. Model MB-CBA Memiliki Banyak Parameter
- CNN layers: 70 → 32 → 64 channels
- BiLSTM dengan attention
- Total parameter: ~150,000-200,000
- Setiap forward + backward membutuhkan waktu signifikan
Estimasi Waktu:
- Perkiraan kasar:
- 1 batch: ~0.01-0.03 detik (dengan GPU)
- 656,250 batch × 0.02 detik = 13,125 detik ≈ 3.6 jam
- Dengan MAML/attention, bisa 2-3x lebih lambat

#Solusi 1: Kurangi Jumlah Data Training (Paling Efektif)

In [ ]:
# Tambahkan di awal cell 8, sebelum LOSO loop
def reduce_dataset(X_subjects, y_subjects, cfg, max_samples_per_class=3000):
    """Sampling data untuk mempercepat training"""
    reduced_X, reduced_y = [], []
    for sub_X, sub_y in zip(X_subjects, y_subjects):
        sampled_X, sampled_y = [], []
        for c in range(cfg['n_classes']):
            idx = np.where(sub_y == c)[0]
            if len(idx) > max_samples_per_class:
                idx = np.random.choice(idx, max_samples_per_class, replace=False)
            sampled_X.append(sub_X[idx])
            sampled_y.append(sub_y[idx])
        reduced_X.append(np.concatenate(sampled_X, axis=0))
        reduced_y.append(np.concatenate(sampled_y, axis=0))
    return reduced_X, reduced_y

# Gunakan sebelum LOSO
X_subjects_small, y_subjects_small = reduce_dataset(X_subjects, y_subjects, CFG, max_samples_per_class=3000)
# Dari 35,980 → 9,000 per subjek (reduce 75% data)

# Ini LOSO Original - prosesnya 3-5 jam (jangan running dulu)

In [ ]:
def train_one_fold(model, tr_loader, val_loader, cfg, device, class_w=None):
    crit = nn.CrossEntropyLoss(
        weight=torch.tensor(class_w, dtype=torch.float).to(device) if class_w is not None else None
    )
    opt  = optim.Adam(model.parameters(), lr=cfg['lr'])
    sched = ReduceLROnPlateau(opt, factor=0.5, patience=3)
    best_loss, best_state, patience_cnt = float('inf'), None, 0

    for epoch in range(1, cfg['max_epochs']+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            logits, _ = model(Xb)
            crit(logits, yb).backward()
            opt.step()

        model.eval()
        val_l = []
        with torch.no_grad():
            for Xb, yb in val_loader:
                logits, _ = model(Xb.to(device))
                val_l.append(crit(logits, yb.to(device)).item())
        vl = np.mean(val_l)
        sched.step(vl)

        if vl < best_loss:
            best_loss  = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= cfg['patience']:
                print(f'    Early stop epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model

def evaluate_fold(model, loader, device):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            logits, _ = model(Xb.to(device))
            all_pred.extend(logits.argmax(1).cpu().numpy())
            all_true.extend(yb.numpy())
    return np.array(all_true), np.array(all_pred)


In [ ]:
results   = []
all_true  = []
all_pred  = []

for test_sub in range(CFG['n_subjects']):
    print(f'\n{"="*55}')
    print(f'  FOLD {test_sub+1}/{CFG["n_subjects"]}  —  Test: Subject {test_sub+1}')
    print('='*55)

    # ── Split LOSO ────────────────────────────────────────
    X_test = torch.tensor(X_subjects[test_sub])
    y_test = torch.tensor(y_subjects[test_sub])

    X_tr = torch.tensor(np.concatenate([X_subjects[i] for i in range(CFG['n_subjects']) if i!=test_sub]))
    y_tr = torch.tensor(np.concatenate([y_subjects[i] for i in range(CFG['n_subjects']) if i!=test_sub]))

    # Val split 20% dari train
    n_val = int(0.2 * len(X_tr))
    idx   = torch.randperm(len(X_tr))
    X_val, y_val  = X_tr[idx[:n_val]], y_tr[idx[:n_val]]
    X_tr2, y_tr2  = X_tr[idx[n_val:]], y_tr[idx[n_val:]]

    print(f'  Train:{len(X_tr2)} | Val:{len(X_val)} | Test:{len(X_test)}')

    # Class weights
    counts  = np.bincount(y_tr2.numpy(), minlength=3)
    class_w = (1.0 / (counts+1e-8))
    class_w = (class_w / class_w.sum() * 3).tolist()

    # DataLoader
    tr_loader  = DataLoader(TensorDataset(X_tr2, y_tr2),
                            batch_size=CFG['batch_size'], shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val),
                            batch_size=CFG['batch_size'])
    te_loader  = DataLoader(TensorDataset(X_test, y_test),
                            batch_size=CFG['batch_size'])

    # Training
    model = MBCBA(CFG).to(device)
    model = train_one_fold(model, tr_loader, val_loader, CFG, device, class_w)

    # Simpan model
    torch.save(model.state_dict(),
               f'{CFG["save_dir"]}/mbcba_fold{test_sub+1}.pt')

    # Evaluasi
    yt, yp = evaluate_fold(model, te_loader, device)
    all_true.extend(yt); all_pred.extend(yp)

    acc = accuracy_score(yt, yp) * 100
    wf1 = f1_score(yt, yp, average='weighted') * 100
    mf1 = f1_score(yt, yp, average='macro')    * 100
    pf1 = f1_score(yt, yp, average=None)       * 100

    print(f'\n  Accuracy    : {acc:.2f}%')
    print(f'  Weighted F1 : {wf1:.2f}%')
    print(f'  Macro F1    : {mf1:.2f}%')
    for i, name in enumerate(CFG['class_names']):
        print(f'  F1 {name:<12}: {pf1[i]:.2f}%')

    results.append({'fold':test_sub+1,'acc':acc,'wf1':wf1,'mf1':mf1,'pf1':pf1})



  FOLD 1/5  —  Test: Subject 1


## 9. Ringkasan LOSO

In [ ]:
accs = [r['acc'] for r in results]
wf1s = [r['wf1'] for r in results]
mf1s = [r['mf1'] for r in results]
pf1s = np.array([r['pf1'] for r in results])

print('='*55)
print('  RINGKASAN LOSO — MB-CBA')
print('='*55)
print(f'  Accuracy    : {np.mean(accs):.2f}% ± {np.std(accs):.2f}%')
print(f'  Weighted F1 : {np.mean(wf1s):.2f}% ± {np.std(wf1s):.2f}%')
print(f'  Macro F1    : {np.mean(mf1s):.2f}% ± {np.std(mf1s):.2f}%')
print()
for i, name in enumerate(CFG['class_names']):
    print(f'  F1 {name:<12}: {pf1s[:,i].mean():.2f}% ± {pf1s[:,i].std():.2f}%')

all_true_arr = np.array(all_true)
all_pred_arr = np.array(all_pred)
print('\n  Classification Report (semua fold):')
print(classification_report(all_true_arr, all_pred_arr,
                             target_names=CFG['class_names'], digits=4))


## 10. Visualisasi Hasil

In [ ]:
# Bar chart per fold
folds = [f'S{r["fold"]}' for r in results]
x     = np.arange(len(folds))
w     = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
b1 = ax.bar(x-w/2, accs, w, label='Accuracy',    color='#378ADD', alpha=0.85)
b2 = ax.bar(x+w/2, wf1s, w, label='Weighted F1', color='#1D9E75', alpha=0.85)

ax.axhline(np.mean(accs), color='#378ADD', ls='--', lw=1.2, alpha=0.7)
ax.axhline(np.mean(wf1s), color='#1D9E75', ls='--', lw=1.2, alpha=0.7)

for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(folds)
ax.set_ylabel('Score (%)'); ax.set_ylim(0, 110)
ax.set_title('Per-fold LOSO performance — MB-CBA', fontsize=12)
ax.set_xlabel('Test Subject (LOSO fold)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('fold_results.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion matrix (normalized)
cm     = confusion_matrix(all_true_arr, all_pred_arr)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            vmin=0, vmax=100,
            xticklabels=CFG['class_names'],
            yticklabels=CFG['class_names'],
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted label', fontsize=11)
ax.set_ylabel('True label',      fontsize=11)
ax.set_title('Normalized confusion matrix (%)', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Training curve (fold terakhir sebagai contoh)
# Untuk melihat semua fold, simpan history di dalam train_one_fold
print('Model disimpan di:', CFG['save_dir'])
print('Plot PDF disimpan: fold_results.pdf, confusion_matrix.pdf')
